# Branch `scalpel-annotation`: what changed and what it does to the numbers

Read-only diagnostic. Re-uses an existing MapMyCells JSON, never re-runs MapMyCells, and
writes only into `cell_type_annotation/_scalpel_check/`.

Section A inlines **both** implementations, `legacy_*` (as on `main`) and `branch_*` (as
on this branch), so the notebook runs against any checkout, including a cluster copy that
has not pulled the branch yet. Only the untouched helpers come from the installed
package. A compatibility cell reports whether the installed copy already matches.

| § | Change | Default behaviour |
|---|---|---|
| B | `MAD_low` (doubleMAD) instead of symmetric MAD, per SCALPEL | **changed** |
| C | cluster label documented as an enrichment argmax, `min_share` floor added | unchanged at `min_share=0` |
| D | `score_delta` guard now also applies when the label is `Undefined` / `Mixed` | **changed**, bug fix |
| E | `Neurons-Dopa-Gaba` alias no longer produces an unmappable label | **changed**, bug fix |

**Only B is about matching SCALPEL.** The paper applies the doubleMAD rule per
**supertype** ("of the 1201 supertypes, more cells were retained in 999, fewer in 192,
and no change in 10") and **removes** the failing cells.

Sections C and D are this pipeline's own cluster-based revision, and it stays.
MapMyCells scores every cell independently, so its labels are noisy at single-cell level
in a targeted panel. Taking a consensus over a fine Leiden cluster, then sanity-checking
that consensus against marker-gene scores, is exactly the denoising MMC does not do for
you. The question here was never whether to revise at cluster level, only whether the two
rules that implement it do what their docstrings say.

In [ ]:
# ruff: noqa
import os, sys, json, logging, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
from scipy.stats import median_abs_deviation
from spatialdata import read_zarr

REPO = "/dss/dsshome1/0C/ra98gaq/Git/cellseg-benchmark-annot"   # <- your checkout
sys.path.insert(1, REPO)
import cellseg_benchmark.cell_annotation_utils as anno_utils
from cellseg_benchmark._constants import cell_type_colors

# Only these are used from the package; none of them changed on this branch.
for fn in ["process_mapmycells_output", "group_cell_types", "create_mixed_cell_types",
           "process_adata", "score_cell_types"]:
    assert hasattr(anno_utils, fn), f"{fn} missing from {REPO}"

warnings.filterwarnings("ignore")
logger = logging.getLogger("scalpel_check")
logger.setLevel(logging.INFO)
if not logger.handlers:
    h = logging.StreamHandler()
    h.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s]: %(message)s"))
    logger.addHandler(h)
pd.set_option("display.width", 160)

In [ ]:
class Args:
    sample_name = "htra1_s3_r0"
    seg_method = "Proseg_3D_Cellpose_1_nuclei_model"
    data_dir = "/dss/dssfs03/pn52re/pn52re-dss-0001/cellseg-benchmark"
    mad_factor = 3.0
    leiden_res = 10.0


args = Args()

method_path = Path(args.data_dir, "samples", args.sample_name, "results", args.seg_method)
annotation_path = method_path / "cell_type_annotation"
out_path = annotation_path / "_scalpel_check"
out_path.mkdir(parents=True, exist_ok=True)

json_path = max(
    annotation_path.glob(f"mapmycells_out/*MapMyCells_{args.sample_name}_{args.seg_method}.json"),
    key=os.path.getmtime,
)
logger.info("Using %s", json_path.name)

## A. Both implementations, inlined

`legacy_*` is `cell_annotation_utils.py` at `main`; `branch_*` is the same file on this
branch. Logging stripped from both. Inlining them keeps the notebook runnable on a
checkout that has not pulled the branch, and keeps the comparison honest: the two rules
run in the same process on the same arrays.

In [ ]:
def legacy_mark_low_quality_mappings(metadata, target_column, mad_factor, level):
    """Symmetric two-sided MAD, no small-group or MAD==0 guard."""
    level_key = f"{target_column}_{level}"
    cor_key = f"{target_column}_cor_{level}"
    out_key = f"{target_column}_{level}_incl_low_quality"
    mask = metadata.groupby(level_key)[cor_key].transform(
        lambda x: x < (x.median() - mad_factor * median_abs_deviation(x, nan_policy="omit"))
    )
    metadata[out_key] = metadata[level_key]
    metadata.loc[mask.fillna(False).astype(bool), out_key] = "Undefined"


def legacy_assign_cell_types_to_clusters(adata, leiden_col, cell_type_col, min_cells=100):
    """Crosstab normalised by the per-cell-type totals, then argmax across cell types."""
    ct = pd.crosstab(adata.obs[leiden_col], adata.obs[cell_type_col],
                     margins=True, margins_name="Total")
    ct = ct.loc[:, ct.loc["Total"] >= min_cells]
    norm = ct.div(ct.loc["Total"], axis=1) * 100
    return norm.drop(index="Total", columns="Total").idxmax(axis=1).to_dict(), norm


def legacy_mmc_to_score_dict(adata):
    """Keeps the marker-table name AND its MMC alias as separate keys."""
    scored = [c.replace("score_", "") for c in adata.obs.columns if c.startswith("score_")]
    d = {ct: ct for ct in scored}
    d.update({"Neurons-Dopa": "Neurons-Dopa-Gaba"})
    return d


def legacy_assign_final_cell_types(
    adata, cluster_labels_dict, mmc_to_score_dict, leiden_col, out_col,
    score_high_threshold=0.5, score_low_threshold=0.25, score_delta=0.2,
):
    """`delta` is NaN when the cluster label has no marker score, and NaN means reassign."""
    adata.obs[out_col] = pd.Series(pd.NA, index=adata.obs.index, dtype=object)
    for cluster in adata.obs[leiden_col].unique():
        assigned_cell_type = cluster_labels_dict.get(cluster, "Undefined")
        mask = adata.obs[leiden_col] == cluster
        all_scores = {
            ct: (adata.obs.loc[mask, f"score_{sfx}"].mean()
                 if f"score_{sfx}" in adata.obs.columns else np.nan)
            for ct, sfx in mmc_to_score_dict.items()
        }
        valid = {k: v for k, v in all_scores.items() if pd.notna(v)}
        if not valid:
            adata.obs.loc[mask, out_col] = "Undefined"
            continue
        highest_cell_type = max(valid, key=valid.get)
        highest_score = valid[highest_cell_type]
        assigned_score = valid.get(assigned_cell_type, np.nan)

        if highest_score >= score_high_threshold:
            delta = (highest_score - assigned_score) if not np.isnan(assigned_score) else np.nan
            adata.obs.loc[mask, out_col] = (
                highest_cell_type if (np.isnan(delta) or delta > score_delta)
                else assigned_cell_type
            )
        elif all(v < score_low_threshold for v in valid.values()):
            adata.obs.loc[mask, out_col] = "Undefined"
        else:
            adata.obs.loc[mask, out_col] = assigned_cell_type
    return adata

In [ ]:
def lower_mad(x, scale=1.0):
    """MAD_low of the doubleMAD: spread of the at-or-below-median half only."""
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    if x.size == 0:
        return np.nan
    median = np.median(x)
    return np.median(np.abs(x[x <= median] - median)) / scale


def branch_mark_low_quality_mappings(metadata, target_column, mad_factor, level,
                                     min_group_cells=10):
    """MAD_low, with small-group and MAD==0 guards."""
    level_key = f"{target_column}_{level}"
    cor_key = f"{target_column}_cor_{level}"
    out_key = f"{target_column}_{level}_incl_low_quality"

    def _flag(x):
        mad = lower_mad(x)
        if x.notna().sum() < min_group_cells or not np.isfinite(mad) or mad == 0:
            return pd.Series(False, index=x.index)
        return x < (x.median() - mad_factor * mad)

    mask = metadata.groupby(level_key)[cor_key].transform(_flag)
    metadata[out_key] = metadata[level_key]
    metadata.loc[mask.fillna(False).astype(bool), out_key] = "Undefined"


def branch_assign_cell_types_to_clusters(adata, leiden_col, cell_type_col,
                                         min_cells=100, min_share=0.0):
    """Enrichment argmax (within-cluster share / sample-wide share), with a purity floor."""
    counts = pd.crosstab(adata.obs[leiden_col], adata.obs[cell_type_col])
    counts = counts.loc[:, counts.sum(axis=0) >= min_cells]

    within = counts.div(counts.sum(axis=1), axis=0)
    overall = counts.sum(axis=0) / counts.to_numpy().sum()
    enrichment = within.div(overall, axis=1).where(within >= min_share)

    assigned = within.idxmax(axis=1)                       # majority fallback
    has_candidate = enrichment.notna().any(axis=1)
    assigned[has_candidate] = enrichment[has_candidate].idxmax(axis=1)
    return assigned.to_dict(), within


def branch_mmc_to_score_dict(adata, valid_labels):
    """Aliases renamed, not added; labels outside cell_type_colors dropped."""
    score_to_mmc_name = {"Neurons-Dopa-Gaba": "Neurons-Dopa"}
    scored = [c.replace("score_", "", 1) for c in adata.obs.columns if c.startswith("score_")]
    d = {score_to_mmc_name.get(ct, ct): ct for ct in scored}
    return {k: v for k, v in d.items() if k in valid_labels}


def branch_assign_final_cell_types(
    adata, cluster_labels_dict, mmc_to_score_dict, leiden_col, out_col,
    score_high_threshold=0.5, score_low_threshold=0.25, score_delta=0.2,
):
    """When the cluster label has no marker score, the guard falls back to the runner-up."""
    adata.obs[out_col] = pd.Series(pd.NA, index=adata.obs.index, dtype=object)
    for cluster in adata.obs[leiden_col].unique():
        assigned_cell_type = cluster_labels_dict.get(cluster, "Undefined")
        mask = adata.obs[leiden_col] == cluster
        all_scores = {
            ct: (adata.obs.loc[mask, f"score_{sfx}"].mean()
                 if f"score_{sfx}" in adata.obs.columns else np.nan)
            for ct, sfx in mmc_to_score_dict.items()
        }
        valid = {k: v for k, v in all_scores.items() if pd.notna(v)}
        if not valid:
            adata.obs.loc[mask, out_col] = "Undefined"
            continue

        ranked = sorted(valid.items(), key=lambda kv: kv[1], reverse=True)
        highest_cell_type, highest_score = ranked[0]
        runner_up_score = ranked[1][1] if len(ranked) > 1 else -np.inf
        assigned_score = valid.get(assigned_cell_type, np.nan)
        reference_score = assigned_score if not np.isnan(assigned_score) else runner_up_score

        if highest_score >= score_high_threshold:
            adata.obs.loc[mask, out_col] = (
                highest_cell_type if (highest_score - reference_score) > score_delta
                else assigned_cell_type
            )
        elif all(v < score_low_threshold for v in valid.values()):
            adata.obs.loc[mask, out_col] = "Undefined"
        else:
            adata.obs.loc[mask, out_col] = assigned_cell_type
    return adata

In [ ]:
# Does the installed checkout already carry the branch?
on_branch = hasattr(anno_utils, "_lower_mad")
print(f"{REPO}\n  -> {'branch code present' if on_branch else 'still on main (pre-branch)'}")

if on_branch:
    rng = np.random.default_rng(0)
    probe = np.concatenate([rng.normal(0.8, 0.03, 500), rng.normal(0.4, 0.1, 50)])
    assert np.isclose(anno_utils._lower_mad(probe), lower_mad(probe))
    print("  -> installed _lower_mad matches the inlined branch version")
else:
    print("  -> the notebook uses its own branch_* definitions, so it runs either way.")
    print("     Pull the branch there before running the pipeline itself.")

## B. MAD variant and taxonomy level

First: which levels survive `--drop_level CCN20230722_SUPT`.

In [ ]:
with open(json_path, "rb") as src:
    mmc_raw = anno_utils.process_mapmycells_output(json.load(src))

levels_present = [lvl for lvl in ["CLAS", "SUBC", "SUPT", "CLUS"]
                  if f"allen_cor_{lvl}" in mmc_raw.columns]
print("levels in this JSON:", levels_present)
print("groups per level   :", {lvl: mmc_raw[f"allen_{lvl}"].nunique() for lvl in levels_present})
print()
print("SCALPEL groups by supertype (1201 in the ABC taxonomy). The pipeline's downstream")
print("flag comes from SUBC. If SUPT is absent above, run_mapmycells() drops it and CLUS")
print("is the closest available level: say that in the methods rather than 'supertype'.")

In [ ]:
rows_B = []
for level in levels_present:
    for name, fn in [("symmetric (main)", legacy_mark_low_quality_mappings),
                     ("MAD_low (branch)", branch_mark_low_quality_mappings)]:
        tmp = mmc_raw[[f"allen_{level}", f"allen_cor_{level}"]].copy()
        fn(tmp, target_column="allen", mad_factor=args.mad_factor, level=level)
        flagged = tmp[f"allen_{level}_incl_low_quality"] == "Undefined"
        rows_B.append({"level": level, "rule": name, "pct_flagged": 100 * flagged.mean()})
        if level == "SUBC":
            globals()["flag_sym" if fn is legacy_mark_low_quality_mappings else "flag_low"] = flagged

summary_B = pd.DataFrame(rows_B).pivot(index="level", columns="rule", values="pct_flagged")
summary_B = summary_B.loc[levels_present].round(2)
summary_B["delta_pp"] = (summary_B["MAD_low (branch)"] - summary_B["symmetric (main)"]).round(2)
summary_B

`delta_pp` should be **negative** where the correlation distribution is left-skewed: the
long lower tail inflates `MAD_low`, pushing the threshold down so fewer cells are
flagged. SCALPEL reports the same direction against a flat 0.5 cutoff (more cells
retained in 999 of 1201 supertypes). A positive `delta_pp` means that level is
right-skewed and is worth looking at before trusting the change.

In [ ]:
print(pd.crosstab(flag_sym, flag_low, rownames=["symmetric"], colnames=["MAD_low"]), "\n")

level = "SUBC"
g = mmc_raw.groupby(f"allen_{level}")[f"allen_cor_{level}"]
thr = pd.DataFrame({
    "n": g.size(),
    "symmetric": g.median() - args.mad_factor * g.apply(lambda s: median_abs_deviation(s, nan_policy="omit")),
    "MAD_low": g.median() - args.mad_factor * g.apply(lower_mad),
}).nlargest(30, "n")

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(range(len(thr)), thr["symmetric"], "o-", label="symmetric (main)")
ax.plot(range(len(thr)), thr["MAD_low"], "o-", label="MAD_low (branch)")
ax.set_xticks(range(len(thr)))
ax.set_xticklabels(thr.index, rotation=90, fontsize=6)
ax.set_ylabel("correlation threshold")
ax.set_title(f"{level}: per-group cutoff, 30 largest groups (lower = fewer cells flagged)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(out_path / "B_mad_variants.png", dpi=150)
plt.show()

Not implemented on this branch: SCALPEL's bimodal handling ("we determined the local
minimum between both peaks and discarded the lower distribution"). Worth adding only if
the per-group correlation histograms actually look bimodal at the level you settle on.

## C. The cluster label

First the equivalence: `min_share=0.0` must reproduce `main` exactly.

In [ ]:
# full pipeline up to Leiden, using the branch code
mmc = mmc_raw.copy()
for lvl in levels_present:
    for prefix in ["allen", "allen_runner_up_1", "allen_runner_up_2"]:
        branch_mark_low_quality_mappings(mmc, target_column=prefix,
                                         mad_factor=args.mad_factor, level=lvl)
for prefix in ["allen", "allen_runner_up_1", "allen_runner_up_2"]:
    for suffix in ["SUBC", "SUBC_incl_low_quality"]:
        mmc[f"{prefix}_{suffix}"] = anno_utils.group_cell_types(mmc[f"{prefix}_{suffix}"])
mmc = mmc.merge(anno_utils.create_mixed_cell_types(df=mmc, diff_threshold=0.5),
                left_index=True, right_index=True, how="left")

adata = read_zarr(method_path / "sdata.zarr")["table"]
adata = adata[:, ~adata.var_names.str.startswith("Blank")]
adata.obsm["allen_cell_type_mapping"] = mmc.loc[adata.obs.index]
adata = anno_utils.process_adata(adata=adata, seg_method=args.seg_method, logger=logger)

leiden_col = f"leiden_res{args.leiden_res}".replace(".", "_")
if leiden_col not in adata.obs:
    sc.tl.leiden(adata, key_added=leiden_col, resolution=args.leiden_res)
print(adata.obs[leiden_col].nunique(), "clusters,", f"{len(adata):,} cells")

In [ ]:
key = "cell_type_mmc_raw"
legacy_labels, _ = legacy_assign_cell_types_to_clusters(adata, leiden_col, key)
branch_labels, within = branch_assign_cell_types_to_clusters(
    adata, leiden_col, key, min_share=0.0
)
assert branch_labels == legacy_labels, "min_share=0 must reproduce main's labels"
print(f"min_share=0.0 reproduces all {len(legacy_labels)} cluster labels from main exactly.")

n_cells = adata.obs[leiden_col].value_counts()
won_share = pd.Series({c: within.loc[c, t] for c, t in branch_labels.items()})
print("\nshare of the cluster held by its winning cell type:")
print(won_share.describe(percentiles=[.01, .05, .10, .25, .5]).round(3).to_string())
print(f"\nclusters won by a type holding <10% of the cluster: {(won_share < 0.10).sum()}"
      f" ({n_cells[won_share[won_share < 0.10].index].sum():,} cells)")

In [ ]:
grid = [0.0, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50]
labels_by_floor, rows_C = {}, []
for f in grid:
    d, _ = branch_assign_cell_types_to_clusters(adata, leiden_col, key, min_share=f)
    labels_by_floor[f] = d
    changed = [c for c in d if d[c] != legacy_labels[c]]
    moved = int(n_cells[changed].sum()) if changed else 0
    rows_C.append({"min_share": f, "clusters_changed": len(changed),
                   "cells_changed": moved, "pct_cells": 100 * moved / len(adata)})
calib = pd.DataFrame(rows_C).set_index("min_share").round(2)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(won_share, bins=50)
ax[0].axvline(0.10, color="r", ls="--", lw=1, label="min_share = 0.10")
ax[0].set_xlabel("share of the cluster held by the winning type")
ax[0].set_ylabel("clusters")
ax[0].set_title("cluster purity at the winning label")
ax[0].legend(fontsize=8)
ax[1].plot(calib.index, calib.pct_cells, "o-")
ax[1].set_xlabel("min_share")
ax[1].set_ylabel("% of cells relabelled vs main")
ax[1].set_title("sensitivity to the floor")
plt.tight_layout()
plt.savefig(out_path / "C_min_share_calibration.png", dpi=150)
plt.show()
calib

Pick the floor from the left panel, not the right one. If cluster purity is bimodal with
a clean gap, put the floor in the gap. If it is a single mass well above any plausible
floor, `main` was already fine on this data and `min_share` is just insurance.

In [ ]:
MIN_SHARE = 0.10        # <- set from the plot above, pass to the script as --min_share

cmp = pd.DataFrame({"main": pd.Series(legacy_labels),
                    "branch": pd.Series(labels_by_floor[MIN_SHARE])})
cmp["n_cells"] = n_cells
cmp["won_share_main"] = won_share.round(3)
cmp["changed"] = cmp.main != cmp.branch
moved_cells = cmp.loc[cmp.changed, "n_cells"].sum()
print(f"clusters relabelled: {cmp.changed.sum()} / {len(cmp)}")
print(f"cells relabelled   : {moved_cells:,} / {len(adata):,} ({100 * moved_cells / len(adata):.1f}%)")
cmp[cmp.changed].sort_values("n_cells", ascending=False).head(20)

In [ ]:
moved = (cmp.loc[cmp.changed].groupby(["main", "branch"], observed=True)["n_cells"].sum()
         .sort_values(ascending=False).rename("cells").reset_index())
per_type = pd.DataFrame({"lost": moved.groupby("main")["cells"].sum(),
                         "gained": moved.groupby("branch")["cells"].sum()}).fillna(0).astype(int)
per_type["net"] = per_type.gained - per_type.lost
print(moved.head(15).to_string(index=False), "\n")
per_type.sort_values("net")

Types with a large negative `net` are the ones the unfloored enrichment over-assigns, and
they should mostly be the rare entries of `cell_type_colors` (Tanycytes, Choroid-Plexus,
ABCs, Neurons-Granule-Immature). Small `net` everywhere means this was a documentation
fix rather than a results change. Whatever you pick applies to all three variants.

## D. The marker-score revision

The cluster label is itself an MMC product, so it is checked against marker expression
before being accepted. That stays. Only the guard deciding when the marker score may win
changed.

In [ ]:
marker_csv = Path(args.data_dir, "misc", "scRNAseq_ref_ABCAtlas_Yao2023Nature",
                  "marker_genes_df", "20250416_cell_type_markers_top50.csv")
marker_df = pd.read_csv(marker_csv)
marker_dict = {c: [g for g in marker_df[c].tolist() if pd.notna(g)]
               for c in marker_df.columns if c != "0"}
marker_dict.pop("Bergmann", None)
adata = anno_utils.score_cell_types(adata, marker_genes_dict=marker_dict,
                                    top_n_genes=50, layer=None, logger=logger)

legacy_map = legacy_mmc_to_score_dict(adata)

branch_map = branch_mmc_to_score_dict(adata, set(cell_type_colors))

print("main   keys not in cell_type_colors:", sorted(set(legacy_map) - set(cell_type_colors)))
print("branch keys not in cell_type_colors:", sorted(set(branch_map) - set(cell_type_colors)))
print(f"\nmain has {len(legacy_map)} keys, branch has {len(branch_map)}. The difference is")
print("the marker-table name kept alongside its MMC alias; under main the argmax can")
print("return it, and the Categorical cast then silently turns those cells into NaN.")

In [ ]:
variants = ["cell_type_mmc_raw", "cell_type_mmc_incl_mixed", "cell_type_mmc_incl_low_quality"]
rows_D = []

for k in variants:
    labels, _ = branch_assign_cell_types_to_clusters(adata, leiden_col, k, min_share=MIN_SHARE)
    adata = legacy_assign_final_cell_types(
        adata, labels, legacy_map, leiden_col, out_col=f"{k}_rev_main",
        score_high_threshold=0.5, score_low_threshold=0.5, score_delta=0.25,
    )
    adata = branch_assign_final_cell_types(
        adata, labels, branch_map, leiden_col, out_col=f"{k}_rev_branch",
        score_high_threshold=0.5, score_low_threshold=0.5, score_delta=0.25,
    )
    flagged = adata.obs[k].isin(["Undefined", "Mixed"])
    rows_D.append({
        "variant": k,
        "flagged upstream": int(flagged.sum()),
        "still flagged, main": int(adata.obs[f"{k}_rev_main"].isin(["Undefined", "Mixed"]).sum()),
        "still flagged, branch": int(adata.obs[f"{k}_rev_branch"].isin(["Undefined", "Mixed"]).sum()),
        "labels differ": int((adata.obs[f"{k}_rev_main"] != adata.obs[f"{k}_rev_branch"]).sum()),
    })

summary_D = pd.DataFrame(rows_D).set_index("variant")
summary_D

In [ ]:
# the clusters the guard is actually about: label has no marker score of its own
k = "cell_type_mmc_incl_low_quality"
labels, _ = branch_assign_cell_types_to_clusters(adata, leiden_col, k, min_share=MIN_SHARE)
means = adata.obs.groupby(leiden_col, observed=True)[list(branch_map.values())].mean()
means.columns = list(branch_map.keys())

rows = []
for c, lab in labels.items():
    s = means.loc[c].dropna().sort_values(ascending=False)
    if s.empty or lab in s.index:
        continue
    rows.append({"cluster": c, "n": int((adata.obs[leiden_col] == c).sum()), "label": lab,
                 "top1": s.index[0], "top1_score": round(s.iloc[0], 3),
                 "top2_score": round(s.iloc[1], 3) if len(s) > 1 else np.nan,
                 "margin": round(s.iloc[0] - s.iloc[1], 3) if len(s) > 1 else np.nan,
                 "main": adata.obs.loc[adata.obs[leiden_col] == c, f"{k}_rev_main"].iloc[0],
                 "branch": adata.obs.loc[adata.obs[leiden_col] == c, f"{k}_rev_branch"].iloc[0]})

blind = pd.DataFrame(rows).set_index("cluster")
print(f"{len(blind)} clusters carry a label with no marker score of its own "
      f"({blind.n.sum():,} cells). Under main, every one with top1 >= 0.5 was reassigned "
      f"regardless of margin.")
blind.sort_values("n", ascending=False).head(25)

Two things to look for in that table:

- `top1_score` just over 0.5 with `margin` near zero: `main` relabelled the cluster on
  evidence that cannot separate the top two candidates, erasing the upstream QC flag.
  The branch keeps it flagged.
- `top1_score` around 0.55 with `margin` well above 0.25: clean evidence, and both keep
  the reassignment. A flat stricter cutoff for Undefined clusters would have discarded it.

Note `score_low_threshold` is passed as **0.5** by the script, equal to
`score_high_threshold`, which makes the "keep the cluster's MMC label" branch unreachable:
every cluster is either reassigned or set to Undefined. The function default is 0.25.
Left unchanged on this branch because lowering it changes results.

## E. Labels that silently become NaN

In [ ]:
produced = set()
for k in variants:
    for sfx in ["_rev_main", "_rev_branch"]:
        produced |= set(pd.Series(adata.obs[f"{k}{sfx}"]).dropna().unique())
print("labels produced but absent from cell_type_colors:",
      sorted(produced - set(cell_type_colors)) or "none")

for k in variants:
    bad_main = (~adata.obs[f"{k}_rev_main"].isin(cell_type_colors)).sum()
    bad_branch = (~adata.obs[f"{k}_rev_branch"].isin(cell_type_colors)).sum()
    print(f"  {k:<35} main {bad_main:>7,} cells | branch {bad_branch:>7,} cells")
print("\nCells in that column are turned into NaN by the Categorical cast in")
print("revise_annotations() and filled as 'Low-Read-Cells' by add_cell_type_annotation().")

In [ ]:
summary_B.to_csv(out_path / "B_mad_variants.csv")
calib.to_csv(out_path / "C_min_share_calibration.csv")
cmp.to_csv(out_path / "C_cluster_labels.csv")
summary_D.to_csv(out_path / "D_revision_rules.csv")
blind.to_csv(out_path / "D_unscored_label_clusters.csv")
print("written to", out_path)